# Level 5 — Evaluasi, error analysis, dan ketidakpastian (Tahap F proyek)

> **Data sintetis.** Skor tidak membuktikan kelayakan model untuk tanggap darurat nyata
> (lihat `../disaster-classification/DATA_CARD.md`). Tidak ada ambang produksi yang ditetapkan di sini.

Model rujukan: Linear SVM (dipilih dari F1 makro validasi silang Level 4, **bukan** dari set tulis-tangan).
Prasyarat: `pip install -r requirements-ml.txt`.

In [1]:
import sys
import warnings
from pathlib import Path

# Notebook ini berada di examples/<folder-level>/. Paket nlp_hub ada di examples/.
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", message="'count' is passed as positional argument")

## 1. Data, model rujukan, dan prediksi

In [2]:
import numpy as np
import pandas as pd
from nlp_hub.proyek import model_rujukan, siapkan

d = siapkan()
model = model_rujukan(d)
skor = {"uji sintetis": model.decision_function(d.uji["text"]), "tulis-tangan": model.decision_function(d.tangan["text"])}
y = {"uji sintetis": d.y_uji, "tulis-tangan": d.y_tangan}
print({k: len(v) for k, v in y.items()})

{'uji sintetis': 112, 'tulis-tangan': 48}


## 2. Confusion matrix dan metrik per kelas
Emergency = 1 (kelas positif). Pada tulis-tangan, perhatikan **FN** (pesan darurat terlewat).

In [3]:
from sklearn.metrics import classification_report, confusion_matrix

for nama in skor:
    p = (skor[nama] > 0).astype(int)
    print(nama)
    print(pd.DataFrame(confusion_matrix(y[nama], p, labels=[1, 0]), index=["sebenarnya E", "sebenarnya N"], columns=["ditebak E", "ditebak N"]))
    print(classification_report(y[nama], p, target_names=["Non-Emergency", "Emergency"], digits=3))

uji sintetis
              ditebak E  ditebak N
sebenarnya E         38          0
sebenarnya N          2         72
               precision    recall  f1-score   support

Non-Emergency      1.000     0.973     0.986        74
    Emergency      0.950     1.000     0.974        38

     accuracy                          0.982       112
    macro avg      0.975     0.986     0.980       112
 weighted avg      0.983     0.982     0.982       112

tulis-tangan
              ditebak E  ditebak N
sebenarnya E         15          9
sebenarnya N          0         24
               precision    recall  f1-score   support

Non-Emergency      0.727     1.000     0.842        24
    Emergency      1.000     0.625     0.769        24

     accuracy                          0.812        48
    macro avg      0.864     0.812     0.806        48
 weighted avg      0.864     0.812     0.806        48



## 3. Ambang: menjelajahi trade-off (bukan memilih)
Tabel ini memakai tulis-tangan untuk **memahami** model. Memilih ambang dari tabel ini lalu melaporkan skornya adalah data snooping.

In [4]:
from nlp_hub.eksperimen import metrik_emergency

baris = []
for t in [-0.6, -0.4, -0.2, 0.0, 0.2]:
    m = metrik_emergency(y["tulis-tangan"], (skor["tulis-tangan"] >= t).astype(int))
    baris.append({"ambang": t, "presisi E": round(m["presisi_e"], 3), "recall E": round(m["recall_e"], 3), "F1 E": round(m["f1_e"], 3)})
pd.DataFrame(baris)

,ambang,presisi E,recall E,F1 E
0,-0.6,0.727,1.000,0.842
1,-0.4,0.880,0.917,0.898
2,-0.2,1.000,0.917,0.957
3,0.0,1.000,0.625,0.769
4,0.2,1.000,0.458,0.629


## 4. Ketidakpastian: selang bootstrap 95% untuk recall Emergency

In [5]:
from nlp_hub.eksperimen import bootstrap_metrik, selang_persentil

for nama in skor:
    p = (skor[nama] > 0).astype(int)
    lo, hi = selang_persentil(bootstrap_metrik(y[nama], p, "recall_e"))
    print(f"{nama:14} recall E = {metrik_emergency(y[nama], p)['recall_e']:.3f}  selang 95% [{lo:.3f}, {hi:.3f}]")

uji sintetis   recall E = 1.000  selang 95% [1.000, 1.000]
tulis-tangan   recall E = 0.625  selang 95% [0.435, 0.810]


## 5. Error analysis: false negative pada tulis-tangan, diurutkan menurut skor

In [6]:
p = (skor["tulis-tangan"] > 0).astype(int)
fn = np.where((d.y_tangan == 1) & (p == 0))[0]
for i in fn[np.argsort(-skor["tulis-tangan"][fn])]:
    print(f"skor={skor['tulis-tangan'][i]:+.2f} | {d.tangan['text'].iloc[i][:80]}")

skor=-0.03 | Mohon info: jalur Puncak km 12 baru saja longsor, 2 mobil keseret ke jurang. sud
skor=-0.04 | Ada bau gas menyengat di gang kami dan tabung bocor berdesis, jangan ada yang ny
skor=-0.06 | Mohon bantuan, ibu hamil 8 bulan terjebak banjir di perumahan Griya Asri blok C,
skor=-0.06 | ledakan di bengkel las, 2 orang terbakar parah, kami nunggu ambulans di depan ge
skor=-0.07 | Astaga dapur tetangga meledak, apinya udah ngejalar ke atap. Siapa punya APAR? J
skor=-0.07 | gempa barusan kerasa banget, saya di lantai 9, tangga penuh orang panik, ada yan
skor=-0.17 | arus sungai bawa anak SD yg main di tepi, org2 lagi ngejar, tlg bantu cari ke ar
skor=-0.46 | pohon tumbang nutup jalan sekaligus nimpa mobil, ada org dlm mobil blm bisa kelu
skor=-0.57 | Banjir bandang!!! jembatan putus, desa kami terisolasi, stok obat diabetes ibu h


**Membaca hasil:** sebagian besar FN berskor dekat 0, tetapi ambang yang dipilih dari data bertemplat tidak berpindah ke
data bergaya lain (lihat halaman *Threshold*). Tuliskan temuan Anda dengan tingkat keyakinan, dan jangan menyimpulkan
kelayakan operasional dari data sintetis.

## Latihan
1. Ganti model rujukan dengan `model_rujukan(d, "nb")` dan ulangi sel 2 dan 4. Apakah selang recall tulis-tangan tumpang tindih dengan SVM?
2. Tambahkan perturbasi dari `nlp_hub.perturbasi.PERTURBASI` dan ukur perubahan recall.